In [0]:
import json

students = [
    {"id": 1, "first": "Jane", "last": "Doe", "grade": 3.8},
    {"id": 2, "first": "John", "last": "Doe", "grade": 3.6},
    {"id": 3, "first": "Paul", "last": "Smith", "grade": 3.9},
    {"id": 4, "first": "Mary", "last": "Jones", "grade": 3.7}
]

with open("students.json", "w") as f:
    json.dump(students, f)

In [0]:
from pyspark.sql import SparkSession
import time

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("CachingPerformanceTest") \
    .getOrCreate()

# Load a sample dataset (modify the path as needed)
students = spark.table("students").select("id", "first", "last", "grade")

# Materialize DataFrame as a temporary table 
students.write.mode("overwrite").saveAsTable("default.temp_students")

# Verify table exists 
if "temp_students" not in [t.name for t in spark.catalog.listTables("default")]:
    raise Exception("Table default.temp_students not found. Create it first!")

# Direct table reads (no reuse)
start_direct = time.time()
count_direct_1 = spark.table("default.temp_students").count()
count_direct_2 = spark.table("default.temp_students").count()  # repeated read
end_direct = time.time()

print(f"[Direct table reads] Counts: {count_direct_1}, {count_direct_2}")
print(f"[Direct table reads] Total time: {end_direct - start_direct:.2f} sec\n")

# Temp variable (logical plan reuse) 
students_temp = spark.table("default.temp_students")  # load once

start_temp = time.time()
count_temp_1 = students_temp.count()
count_temp_2 = students_temp.count()  # reuse variable 
end_temp = time.time()

print(f"[Temp variable] Counts: {count_temp_1}, {count_temp_2}")
print(f"[Temp variable] Total time: {end_temp - start_temp:.2f} sec\n")

# Materialized table reuse (replacement for caching) 
start_materialized = time.time()
count_mat_1 = spark.table("default.temp_students").count()
count_mat_2 = spark.table("default.temp_students").count()  # repeated read
end_materialized = time.time()

print(f"[Materialized table] Counts: {count_mat_1}, {count_mat_2}")
print(f"[Materialized table] Total time: {end_materialized - start_materialized:.2f} sec\n")

# Direct table reads (no reuse) 
# Repartitioned DataFrame (performance tuning)
students_repart = spark.table("default.temp_students").repartition(10)

start_repart = time.time()
count_repart_1 = students_repart.count()
count_repart_2 = students_repart.count()  # repeated action
end_repart = time.time()

print(f"[Repartitioned DataFrame] Counts: {count_repart_1}, {count_repart_2}")
print(f"[Repartitioned DataFrame] Total time: {end_repart - start_repart:.2f} sec\n")

# Cleanup: Drop temporary table 
spark.sql("DROP TABLE IF EXISTS default.temp_students")


[Direct table reads] Counts: 4, 4
[Direct table reads] Total time: 0.89 sec

[Temp variable] Counts: 4, 4
[Temp variable] Total time: 0.96 sec

[Materialized table] Counts: 4, 4
[Materialized table] Total time: 1.01 sec

[Repartitioned DataFrame] Counts: 4, 4
[Repartitioned DataFrame] Total time: 2.39 sec



DataFrame[]

In [0]:
from pyspark.sql.functions import broadcast

small_data = students.limit(10)
joined_data = students.join(broadcast(small_data), "id")
joined_data.show()

large_data = students
joined_data = large_data.join(students, "id")
joined_data.show()


+---+-----+-----+-----+-----+-----+-----+
| id|first| last|grade|first| last|grade|
+---+-----+-----+-----+-----+-----+-----+
|  1| Jane|  Doe|  3.8| Jane|  Doe|  3.8|
|  2| John|  Doe|  3.6| John|  Doe|  3.6|
|  3| Paul|Smith|  3.9| Paul|Smith|  3.9|
|  4| Mary|Jones|  3.7| Mary|Jones|  3.7|
+---+-----+-----+-----+-----+-----+-----+

+---+-----+-----+-----+-----+-----+-----+
| id|first| last|grade|first| last|grade|
+---+-----+-----+-----+-----+-----+-----+
|  1| Jane|  Doe|  3.8| Jane|  Doe|  3.8|
|  2| John|  Doe|  3.6| John|  Doe|  3.6|
|  3| Paul|Smith|  3.9| Paul|Smith|  3.9|
|  4| Mary|Jones|  3.7| Mary|Jones|  3.7|
+---+-----+-----+-----+-----+-----+-----+

